In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve, auc
import os
import pickle

import scanpy as sc 
import json
import re
import pyranges as pr
from cellgrn.utils import enhancer_eval, eval_gene_peak, eval_tf_recovery, eval_tf_recovery_ctx, eval_tf_gene, load_scenic2, load_linger_ctx,load_linger_all,load_thres_grn

/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [ ]:
result_path = "/home/shaliu_fu/multireg/benchmark/output/10X_PBMC/"
# scgpt_grn =  pd.read_csv("/home/shaliu_fu/scGraphGPT/reproducibility_code/output/scgpt_grn_baseline.csv")
dataset_config = json.load(open("/home/shaliu_fu/multireg/benchmark/bench_dataset/10X_PBMC/RawData.json"))

soft_path = {
    "SCENIC+":"scenic2_output",
    "LINGER":"LINGER_output"
}

soft_load = {
    "SCENIC+":load_scenic2,
    "LINGER":load_linger_all,

}

In [ ]:
# 0. load files 
# soft_res = {}
# for soft in soft_path.keys():
#     print(soft)
#     gene_peak_res, grn_res, tf_peak_res = soft_load[soft](result_path, soft_path[soft], dataset_config)
#     soft_res[soft] = {}
#     soft_res[soft]['grn_res'] = grn_res
#     soft_res[soft]['gene_peak_res'] = gene_peak_res
#     soft_res[soft]['tf_peak_res'] = tf_peak_res

# # ctx specific results
# soft_res_ctx = {}
# gene_peak_res, grn_res, tf_peak_res = load_linger_ctx(result_path, "LINGER_output", dataset_config)
# soft_res_ctx["LINGER_ctx"] = {}
# soft_res_ctx["LINGER_ctx"]['grn_res'] = grn_res
# soft_res_ctx["LINGER_ctx"]['gene_peak_res'] = gene_peak_res
# soft_res_ctx["LINGER_ctx"]['tf_peak_res'] = tf_peak_res
# with open("./pbmc_bench_methods.pkl", "wb") as f:
#     pickle.dump(soft_res, f)

# with open("./pbmc_bench_methods_ctx.pkl", "wb") as f:
#     pickle.dump(soft_res_ctx, f)

# load pickle files
with open("./pbmc_bench_methods.pkl", "rb") as f:
    soft_res = pickle.load(f)
with open("./pbmc_bench_methods_ctx.pkl", "rb") as f:
    soft_res_ctx = pickle.load(f)


In [ ]:

for suffix in ["scale2"]:

    soft = f'linger_thres_{suffix}_samp'
    gene_peak_res, grn_res,tf_peak_res = load_thres_grn(f"/home/shaliu_fu/multireg/cellGRN/output/res_pbmc_linger/",
                                                        scale="sample",suffix=suffix,peak_rev=True)
    soft_res[soft] = {}
    soft_res[soft]['grn_res'] = grn_res
    soft_res[soft]['gene_peak_res'] = gene_peak_res
    soft_res[soft]['tf_peak_res'] = tf_peak_res

    soft = f'linger_thres_{suffix}_ctx'
    gene_peak_res, grn_res,tf_peak_res = load_thres_grn(f"/home/shaliu_fu/multireg/cellGRN/output/res_pbmc_linger/",
                                                        scale='celltype',suffix=suffix,peak_rev=True)
    soft_res_ctx[soft] = {}
    soft_res_ctx[soft]['grn_res'] = grn_res
    soft_res_ctx[soft]['gene_peak_res'] = gene_peak_res
    soft_res_ctx[soft]['tf_peak_res'] = tf_peak_res

    soft = f'scenic2_thres_{suffix}_samp'
    gene_peak_res, grn_res,tf_peak_res = load_thres_grn(f"/home/shaliu_fu/multireg/cellGRN/output/res_pbmc_scenic2/",
                                                        scale="sample",suffix=suffix,peak_rev=True)
    soft_res[soft] = {}
    soft_res[soft]['grn_res'] = grn_res
    soft_res[soft]['gene_peak_res'] = gene_peak_res
    soft_res[soft]['tf_peak_res'] = tf_peak_res



outdir = "/home/shaliu_fu/multireg/cellGRN/eval/results/pbmc/"
os.system(f"mkdir -p {outdir}")

0

In [5]:
# 1. enhancer eval
cd4_gold = pd.read_csv("/home/shaliu_fu/multireg/benchmark/datasets/pbmc/10X_PBMC_CD4_STARR.bed",sep='\t',header=None)
cd4_gold['Peak'] = cd4_gold.apply(lambda row:f"{row[0]}:{row[1]}-{row[2]}", axis=1)

In [ ]:
pr_curve = pd.DataFrame()
res_summary = []



for soft in soft_res.keys():
    gene_peak_res = soft_res[soft]['gene_peak_res'].copy()

    if gene_peak_res is not None:
        pr_table,pr_auc,epr,f1  = enhancer_eval(cd4_gold, gene_peak_res,soft)
        pr_curve = pd.concat([pr_curve,pr_table],axis=0)
        res_summary.append([soft,round(pr_auc,5),round(epr,5),round(f1,5)])


for soft in soft_res_ctx.keys():
    gene_peak_res = soft_res_ctx[soft]['gene_peak_res']
    if gene_peak_res is not None:
        gene_peak_res_ = gene_peak_res[gene_peak_res['cell_type']=='CD4 T']
        soft_ = f"{soft}_CD4"
        pr_table,pr_auc,epr,f1  = enhancer_eval(cd4_gold, gene_peak_res_,soft_)
        pr_curve = pd.concat([pr_curve,pr_table],axis=0)
        res_summary.append([soft_,round(pr_auc,5),round(epr,5),round(f1,5)])


res_summary = pd.DataFrame(res_summary)
res_summary.columns = ["method","PRAUC","EPR","F1"]



/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:230: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  label.loc[label > 1] = 1
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:230: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  label.loc[label > 1] = 1
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:230: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#retur

In [ ]:
out_res = res_summary
out_res.to_csv(f"{outdir}/cd4_enhancer_gene_peak_res.csv",index=False,header=True)
pr_curve.to_csv(f"{outdir}/cd4_enhancer_gene_peak_pr_curve.csv",index=False,header=True)


In [ ]:


pr_curve = pd.DataFrame()
res_summary = []


for soft in soft_res.keys():
    tf_peak_res = soft_res[soft]['tf_peak_res']

    if tf_peak_res is not None:
        tf_peak_res = tf_peak_res.nlargest(20000,"Score")
        pr_table,pr_auc,epr,f1  = enhancer_eval(cd4_gold, tf_peak_res,soft)
        pr_curve = pd.concat([pr_curve,pr_table],axis=0)
        res_summary.append([soft,round(pr_auc,5),round(epr,5),round(f1,5)])



for soft in soft_res_ctx.keys():
    tf_peak_res = soft_res_ctx[soft]['tf_peak_res']
    if tf_peak_res is not None:
        tf_peak_res_ = tf_peak_res[tf_peak_res['cell_type']=='CD4 T']
        tf_peak_res_ = tf_peak_res_.nlargest(20000,"Score")
        soft_ = f"{soft}_CD4"
        pr_table,pr_auc,epr,f1  = enhancer_eval(cd4_gold, tf_peak_res_,soft_)
        pr_curve = pd.concat([pr_curve,pr_table],axis=0)
        res_summary.append([soft_,round(pr_auc,5),round(epr,5),round(f1,5)])

    
res_summary = pd.DataFrame(res_summary)
res_summary.columns = ["method","PRAUC","EPR","F1"]



/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:230: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  label.loc[label > 1] = 1
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:230: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  label.loc[label > 1] = 1
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:230: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#retur

In [ ]:
out_res = res_summary
out_res.to_csv(f"{outdir}/cd4_enhancer_tf_peak_res.csv",index=False,header=True)

pr_curve.to_csv(f"{outdir}/cd4_enhancer_tf_peak_pr_curve.csv",index=False,header=True)



In [14]:
from scipy.sparse import load_npz

gene_peak_link = load_npz('/home/shaliu_fu/multireg/multigrn/input_data/all_gene/10X_PBMC/gene_peak_dist_all.npz')
input_gene = [i.rstrip() for i in open("/home/shaliu_fu/multireg/multigrn/input_data/all_gene/10X_PBMC/input_gene.txt")]
input_peak = [i.rstrip() for i in open("/home/shaliu_fu/multireg/multigrn/input_data/all_gene/10X_PBMC/input_peak.txt")]
gene_peak_coo = gene_peak_link.tocoo()
peak_idx = gene_peak_coo.col   # [n_nonzero]
gene_idx = gene_peak_coo.row   # [n_nonzero]

gene_peak_dist = pd.DataFrame({"Gene" : pd.Series(input_gene).values[gene_idx],
                              "Peak" : pd.Series(input_peak).values[peak_idx],
                              "Dist": gene_peak_coo.data
})
gene_peak_dist['Peak'] = gene_peak_dist['Peak'].str.replace(r'^([^-\s]+)-', r'\1:', regex=True)

In [ ]:


ctx_dict = {"CD4 T":"cd4","B":"b","CD8 T":"cd8","NK":"nk"}


res_summary = []


range_summary = pd.DataFrame()
pr_curve = pd.DataFrame()

for ctx in ctx_dict.keys():
    hic_gold = f"/home/shaliu_fu/multireg/benchmark/datasets/pbmc/encode_hic/{ctx_dict[ctx]}_gene_peak_gold.bed"
    gold_pr_region = pr.read_bed(hic_gold)  
    for soft in soft_res.keys():

        gene_peak_res = soft_res[soft]['gene_peak_res']
        # soft_ = f"{soft}_{ctx}"
        if gene_peak_res is not None:
            gene_peak_res_ = pd.merge(gene_peak_res,gene_peak_dist,on=["Gene","Peak"],how="left")

            pr_table,pr_auc,epr,f1, range_res = eval_gene_peak(gold_pr_region, gene_peak_res_,gene_peak_dist, soft)
            pr_table['celltype']=ctx
            pr_curve = pd.concat([pr_curve,pr_table],axis=0)
            res_summary.append([soft,round(pr_auc,5),round(epr,5),round(f1,5),ctx])
            range_res['celltype']=ctx
            range_summary = pd.concat([range_summary,range_res], axis=0)

    for soft in soft_res_ctx.keys():
        gene_peak_res = soft_res_ctx[soft]['gene_peak_res']
        if gene_peak_res is not None:
            gene_peak_res_ = gene_peak_res[gene_peak_res['cell_type']==ctx]
            gene_peak_res_ = pd.merge(gene_peak_res_,gene_peak_dist,on=["Gene","Peak"],how="left")
            # soft_ = f"{soft}_{ctx}"
            # pr_table,pr_auc,epr = enhancer_eval(cd4_gold, gene_peak_res_,soft_)
            pr_table,pr_auc,epr,f1, range_res = eval_gene_peak(gold_pr_region, gene_peak_res_,gene_peak_dist, soft)
            pr_table['celltype']=ctx
            pr_curve = pd.concat([pr_curve,pr_table],axis=0)
            range_res['celltype']=ctx

            res_summary.append([soft,round(pr_auc,5),round(epr,5),round(f1,5),ctx])
            range_summary = pd.concat([range_summary,range_res], axis=0)



res_summary = pd.DataFrame(res_summary)
res_summary.columns = ["method","PRAUC","EPR","F1","celltype"]

/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:300: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['peak_gene'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}_{row[3]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:303: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['Peak'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:307: FutureWarning: Series.__getitem__ treating keys a

In [13]:
pr_curve.head()

,prec,recall,prauc,method,cell-type
0,0.019608,1.000000,0.05082,SCENIC+,CD4 T
1,0.077602,0.057707,0.05082,SCENIC+,CD4 T
2,0.077604,0.057707,0.05082,SCENIC+,CD4 T
3,0.077606,0.057707,0.05082,SCENIC+,CD4 T
4,0.077609,0.057707,0.05082,SCENIC+,CD4 T


In [ ]:
out_res = res_summary
out_res.to_csv(f"{outdir}/pbmc_ctx_hic_res.csv",index=False,header=True)

pr_curve.to_csv(f"{outdir}/pbmc_ctx_hic_pr_curve.csv",index=False,header=True)
range_summary.to_csv(f"{outdir}/pbmc_ctx_hic_range_res.csv",index=False,header=True)

In [ ]:
# epr_summary['cell-type'].unique()

array(['CD4 T', 'B', 'CD8 T', 'NK'], dtype=object)

In [127]:
pr_curve['method'].unique()

array(['SCENIC+', 'LINGER', 'linger_thres_scale2_samp',
       'scenic2_thres_scale2_samp', 'LINGER_ctx',
       'linger_thres_scale2_ctx'], dtype=object)

In [260]:
gold_pr_region.df['Name'].isin(gene_peak_res_['Gene']).sum()

4174

In [ ]:


ctx_dict = {"CD4 T":"CD4T","B":"B","CD8 T":"CD8T","NK":"NK","Mono":"monocyte"}

# ctx = "CD4 T"
# hic_gold = f"/home/shaliu_fu/multireg/benchmark/datasets/pbmc/sc-eQTLGen/{ctx_dict[ctx]}_eQTL_gold.txt"

res_summary = []
range_summary = pd.DataFrame()
pr_curve = pd.DataFrame()

for ctx in ctx_dict.keys():
# for ctx in ['CD4 T']:
    hic_gold = f"/home/shaliu_fu/multireg/benchmark/datasets/pbmc/sc-eQTLGen/{ctx_dict[ctx]}_eQTL_gold.txt"

    gold_pr_region = pr.read_bed(hic_gold)
    sel_gene = gold_pr_region.df['Name']

    for soft in soft_res.keys():
    # for soft in ['FigR']:
        gene_peak_res = soft_res[soft]['gene_peak_res']
    # for soft in spa_res.keys():
    #     gene_peak_res = spa_res[soft]

        if gene_peak_res is not None:
            gene_peak_res_ = pd.merge(gene_peak_res,gene_peak_dist,on=["Gene","Peak"],how="left")
            gene_peak_res_ = gene_peak_res_[gene_peak_res_['Gene'].isin(sel_gene)]
            print(f"{soft}: {gene_peak_res_.shape}")
            pr_table,pr_auc,epr,f1, range_res = eval_gene_peak(gold_pr_region, gene_peak_res_,gene_peak_dist, soft)
            pr_table['celltype'] = ctx
            pr_curve = pd.concat([pr_curve,pr_table],axis=0)

            range_res['celltype'] = ctx
            res_summary.append([soft,round(pr_auc,5),round(epr,5),round(f1,5),ctx])
            range_summary = pd.concat([range_summary,range_res], axis=0)

    for soft in soft_res_ctx.keys():
        gene_peak_res = soft_res_ctx[soft]['gene_peak_res'] 
        if gene_peak_res is not None:
            # if "cell_type" in gene_peak_res.columns.values:
            gene_peak_res_ = gene_peak_res[gene_peak_res['cell_type']==ctx]
            gene_peak_res_ = pd.merge(gene_peak_res_,gene_peak_dist,on=["Gene","Peak"],how="left")

            pr_table,pr_auc,epr,f1, range_res = eval_gene_peak(gold_pr_region, gene_peak_res_,gene_peak_dist, soft)

            pr_table['celltype'] = ctx
            pr_curve = pd.concat([pr_curve,pr_table],axis=0)
            range_res['celltype'] = ctx
            
            res_summary.append([soft,round(pr_auc,5),round(epr,5),round(f1,5),ctx])
            range_summary = pd.concat([range_summary,range_res], axis=0)


res_summary = pd.DataFrame(res_summary)
res_summary.columns = ["method","PRAUC","EPR","F1","celltype"]

SCENIC+: (2213, 4)


/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:300: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['peak_gene'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}_{row[3]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:303: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['Peak'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:307: FutureWarning: Series.__getitem__ treating keys a

LINGER: (884, 4)


/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:300: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['peak_gene'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}_{row[3]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:303: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['Peak'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:307: FutureWarning: Series.__getitem__ treating keys a

linger_thres_scale2_samp: (939, 4)


...

/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:300: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['peak_gene'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}_{row[3]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:303: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['Peak'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:307: FutureWarning: Series.__getitem__ treating keys a

linger_thres_scale2_samp: (940, 4)


/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:300: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['peak_gene'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}_{row[3]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:303: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['Peak'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:307: FutureWarning: Series.__getitem__ treating keys a

scenic2_thres_scale2_samp: (1019, 4)


/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:300: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['peak_gene'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}_{row[3]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:303: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['Peak'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:307: FutureWarning: Series.__getitem__ treating keys a

In [14]:
# gene_peak_dist
range_summary.to_csv(f"{outdir}/pbmc_ctx_eQTL_range_res.csv",index=False,header=True)

In [15]:
out_res = res_summary
out_res.to_csv(f"{outdir}/pbmc_ctx_eQTL_res.csv",index=False,header=True)

pr_curve.to_csv(f"{outdir}/pbmc_ctx_eQTL_pr_curve.csv",index=False,header=True)


In [155]:
df = pr_curve[pr_curve['method'].isin(['LINGER_ctx', 'linger_thres_scale2_ctx'])]
df.head()

,prec,recall,prauc,method,cell-type
0,0.019608,1.000000,0.010096,LINGER_ctx,CD4 T
1,0.000722,0.009009,0.010096,LINGER_ctx,CD4 T
2,0.000722,0.009009,0.010096,LINGER_ctx,CD4 T
3,0.000722,0.009009,0.010096,LINGER_ctx,CD4 T
4,0.000722,0.009009,0.010096,LINGER_ctx,CD4 T


In [ ]:
# epr_summary['cell-type'].unique()

array(['CD4 T', 'B', 'CD8 T', 'NK', 'Mono'], dtype=object)

In [ ]:
with open("/home/shaliu_fu/multireg/multigrn/input_data/gold_dataset/pbmc_gold.pkl", "rb") as f:
    gold_data = pickle.load(f)


tf_gold = {
    "B": [
        "PAX5",    # B细胞谱系决定核心因子
        "EBF1",    # 早期B细胞发育关键因子
        "POU2F2",  # (Oct-2) 调节免疫球蛋白基因表达
        "BCL6",    # 生发中心B细胞标志
        "IRF4"     # 浆细胞分化关键因子
    ],
    "CD4 T": [
        "TCF7",    # (TCF-1) 幼稚/记忆状态维持
        "LEF1",    # 幼稚T细胞标志
        "TBX21",   # (T-bet) Th1亚群标志
        "GATA3",   # Th2亚群标志
        "FOXP3",   # Treg(调节性T细胞)核心标志
        "RORC"     # (RORγt) Th17亚群标志
    ],
    "CD8 T": [
        "RUNX3",   # CD8+ 谱系决定因子
        "EOMES",   # 效应与记忆功能调控
        "TBX21",   # (T-bet) 细胞毒性功能调控
        "PRDM1"    # (Blimp-1) 终末分化效应细胞标志
    ],
    "DC": [
        "TCF4",    # (E2-2) pDC(浆细胞样DC)特异性标志
        "IRF8",    # cDC1 和 pDC 发育关键因子
        "BATF3",   # cDC1 亚型特异性因子
        "IRF4",    # cDC2 亚型相关因子
        "ZEB2"     # 调控DC发育与分化
    ],
    "Mono": [
        "SPI1",    # (PU.1) 髓系发育主控因子
        "CEBPB",   # (C/EBPβ) 非经典单核细胞(CD16+)核心因子
        "MAFB",    # 单核/巨噬细胞分化标志
        "KLF4",    # 经典单核细胞(CD14+)相关因子
        "IRF8"     # 影响单核细胞向DC或巨噬细胞的分化
    ],
    "NK": [
        "EOMES",   # NK细胞发育与成熟关键因子
        "TBX21",   # (T-bet) 调控NK细胞毒性分子表达
        "ID2",     # 抑制T/B谱系，促进NK发育
        "IKZF3"    # (Aiolos) 调节NK细胞功能
    ]
}
tf_knock_gold = gold_data['tf_gene_gold']
tf_baseline = gold_data['tf_gene_baseline'].copy()

tf_baseline = tf_baseline[tf_baseline['TF']!=tf_baseline['Gene']]
tf_baseline = tf_baseline.nlargest(10000,"Score")

In [9]:
# # 第一次计算保存下pickle, 后续不重复计算了，有点慢。
# TF_gene_knock = pd.read_csv("/home/shaliu_fu/multireg/benchmark/datasets/pbmc/TF_gene_gold.txt",sep="\t",index_col=None,header=0)
# TF_gene_knock2 = TF_gene_knock[TF_gene_knock['TF'].isin(all_genes)][TF_gene_knock['Gene'].isin(all_genes)]


# # PBMC_TF_knock_pair = TF_gene_knock.apply(lambda x: f"{x[1]}_{x[2]}",axis=1)

# PBMC_TF_knock_all = TF_gene_knock.apply(lambda x: f"{x[1]}_{x[2]}",axis=1)
# PBMC_TF_knock_pair = TF_gene_knock2.apply(lambda x: f"{x[1]}_{x[2]}",axis=1)

# scRNA_tab = sc.read_h5ad("/home/shaliu_fu/multireg/benchmark/bench_dataset/10X_PBMC/{0}".format(dataset_config['rna_h5ad_filename']))


# common_genes = set(gene for gene in TF_gene_knock['TF'] if gene in scRNA_tab.var_names)
# expression_matrix = scRNA_tab.X.toarray() if hasattr(scRNA_tab.X, "toarray") else scRNA_tab.X
# row_std = np.std(expression_matrix, axis=1)
# expression_matrix = expression_matrix[row_std!=0]
# # 提取感兴趣基因的表达量
# gene_indices = [scRNA_tab.var_names.get_loc(gene) for gene in common_genes]
# selected_genes_expression = expression_matrix[:, gene_indices]  # shape: (cells, len(common_genes))

# expression_matrix_zscore = (expression_matrix - np.mean(expression_matrix, axis=0)) / np.std(expression_matrix, axis=0)
# selected_genes_zscore = (selected_genes_expression - np.mean(selected_genes_expression, axis=0)) / np.std(selected_genes_expression, axis=0)

# # 2. 计算相关性：矩阵乘法（内积）
# correlation_matrix = np.dot(expression_matrix_zscore.T, selected_genes_zscore) / expression_matrix.shape[0]

# # 将结果存储为 DataFrame（行：所有基因，列：感兴趣基因的相关性）
# correlation_df = pd.DataFrame(correlation_matrix, index=scRNA_tab.var_names, columns=list(common_genes))

# melted_df = correlation_df.reset_index().melt(id_vars="index", var_name="TF", value_name="Score")

# # 重命名行名列为 "Gene"
# melted_df.rename(columns={"index": "Gene"}, inplace=True)

# melted_df = melted_df[["TF","Gene","Score"]]
# melted_df.dropna(inplace=True)

# grn_res = melted_df

# overlap_tf = set(grn_res['TF']) & set(TF_gene_knock['TF'])
# # for tf in overlap_tf:
# soft_pred = grn_res[grn_res['TF'].isin(overlap_tf)]

# tf_baseline = soft_pred.copy()
# tf_baseline = tf_baseline[tf_baseline['TF']!=tf_baseline['Gene']]

# gold_data = dict()
# gold_data['tf_gene_baseline'] = tf_baseline
# gold_data['tf_gene_gold'] = PBMC_TF_knock_pair

# with open("/home/shaliu_fu/multireg/multigrn/input_data/gold_dataset/pbmc_gold.pkl", "w") as f:
#     pickle.dump(gold_data,f)

In [ ]:


rec_summary = pd.DataFrame()
rec_ctx = pd.DataFrame() 


tf_rec_ctx, tf_rec_base = eval_tf_recovery(grn_res=tf_baseline,gold_data=tf_gold,label="Pearson",log=False) # ctx dataframe

rec_summary = pd.concat([rec_summary,tf_rec_base],axis=0)

tf_rec_ctx['method'] = "Pearson"
rec_ctx = pd.concat([rec_ctx, tf_rec_ctx],axis=0)

for soft in soft_res.keys():
# for soft in ['FigR']:

# for soft in spa_res.keys():
#     gene_peak_res = spa_res[soft]
    tf_gene_res = soft_res[soft]['grn_res']
    tf_gene_res = tf_gene_res[["TF","Gene","Score"]]
    tf_gene_res = tf_gene_res.groupby(['TF', 'Gene'])['Score'].max().reset_index() 
    # tf_gene_res.drop_duplicates(inplace=True) 
    if tf_gene_res is not None:
        
        tf_gene_res2 = tf_gene_res.nlargest(10000,"Score")
        tf_rec_ctx, tf_rec_res = eval_tf_recovery(grn_res=tf_gene_res2,gold_data=tf_gold,label=soft,log=False) # ctx dataframe
        # rec_summary.append([soft,round(tf_rec_res.values[0][0],5),round(tf_rec_res.values[0][1],5)])
        rec_summary = pd.concat([rec_summary,tf_rec_res],axis=0)
        tf_rec_ctx['method'] = soft
        rec_ctx = pd.concat([rec_ctx, tf_rec_ctx],axis=0)


for soft in soft_res_ctx.keys():

    tf_gene_res = soft_res_ctx[soft]['grn_res']

    if tf_gene_res is not None:        
        # tf_gene_res2 = tf_gene_res.nlargest(10000,"Score")
        tf_rec_ctx, tf_rec_res = eval_tf_recovery_ctx(grn_res_ctx=tf_gene_res,gold_data=tf_gold,label=soft,log=False) # ctx dataframe
        # rec_summary.append([soft,round(tf_rec_res.values[0][0],5),round(tf_rec_res.values[0][1],5)])
        rec_summary = pd.concat([rec_summary,tf_rec_res],axis=0)
        tf_rec_ctx['method'] = soft
        rec_ctx = pd.concat([rec_ctx, tf_rec_ctx],axis=0)

In [18]:
rec_summary.to_csv(f"{outdir}/tf_marker_tf_gene_res.csv",index=True,header=True)
rec_ctx.to_csv(f"{outdir}/tf_marker_tf_gene_res_ctx.csv",index=True,header=True)

In [ ]:

rec_summary = pd.DataFrame()
rec_ctx = pd.DataFrame() 


for soft in soft_res.keys():

    tf_peak_res = soft_res[soft]['tf_peak_res']
    if tf_peak_res is not None:
        tf_peak_res = tf_peak_res[["TF","Peak","Score"]]
        tf_peak_res = tf_peak_res.groupby(['TF', 'Peak'])['Score'].max().reset_index() #只保留最高的。
        
        tf_peak_res2 = tf_peak_res.nlargest(20000,"Score")
        tf_rec_ctx, tf_rec_res = eval_tf_recovery(grn_res=tf_peak_res2,gold_data=tf_gold,label=soft,log=False) # ctx dataframe
        rec_summary = pd.concat([rec_summary,tf_rec_res],axis=0)
        tf_rec_ctx['method'] = soft
        rec_ctx = pd.concat([rec_ctx, tf_rec_ctx],axis=0)


for soft in soft_res_ctx.keys():

    tf_peak_res = soft_res_ctx[soft]['tf_peak_res']
    
    if tf_peak_res is not None: 
        # tf_peak_res = tf_peak_res.groupby(['TF', 'Peak','cell_type'])['Score'].max().reset_index() #只保留最高的。       
        # tf_peak_res2 = tf_peak_res.nlargest(40000,"Score")
        tf_rec_ctx, tf_rec_res = eval_tf_recovery_ctx(grn_res_ctx=tf_peak_res,gold_data=tf_gold,label=soft,log=False) # ctx dataframe
        rec_summary = pd.concat([rec_summary,tf_rec_res],axis=0)
        tf_rec_ctx['method'] = soft
        rec_ctx = pd.concat([rec_ctx, tf_rec_ctx],axis=0)

In [13]:
rec_summary.to_csv(f"{outdir}/tf_marker_tf_peak_res.csv",index=True,header=True)
rec_ctx.to_csv(f"{outdir}/tf_marker_tf_peak_res_ctx.csv",index=True,header=True)

In [54]:
rec_summary

,Best_F1_avg,Best_Precision_avg,Best_Recall_avg,Best_N_avg
method,,,,
SCENIC+,0.0628,0.0419,0.1500,12.3333
LINGER,0.0675,0.0736,0.1778,39.0000
linger_thres_scale2_samp,0.0726,0.0647,0.1611,9.8333
scenic2_thres_scale2_samp,0.1034,0.0913,0.5528,60.6667
LINGER_ctx,0.0948,0.0562,0.3722,40.5000
linger_thres_scale2_ctx,0.2100,0.1918,0.2972,12.6667


In [55]:
tmp = tf_knock_gold['PBMC_TF_knock']
sel_tf = set([i.split("_")[0] for i in tmp])
sel_tf

tf_knock_gold2 = {}
for tfs in sel_tf:
    tf_knock_gold2[tfs] = []

for i in tf_knock_gold['PBMC_TF_knock']:
    tf = i.split("_")[0]
    gene = i.split("_")[1]
    tf_knock_gold2[tf].append(gene)

In [56]:
# tmp = tf_knock_gold['PBMC_TF_knock']
# sel_tf = set([i.split("_")[0] for i in tmp])
tf_baseline = gold_data['tf_gene_baseline'].copy()

pr_curve = pd.DataFrame()
res_summary = []
# 评估tf-gene

for t_gold in tf_knock_gold.keys():
    tf_knock = tf_knock_gold[t_gold]
    
    tf_baseline2 = tf_baseline[tf_baseline["TF"].isin(sel_tf)]
    
    tf_baseline2 = tf_baseline2[tf_baseline2['TF']!=tf_baseline2['Gene']]
    
    

    pr_auc,epr,f1, pr_table = eval_tf_gene(grn_res=tf_baseline2.nlargest(10000,"Score"),gold_data=tf_knock,
                    all_comb=tf_baseline2.shape[0],label="Pearson" ) # 
    # pr_table['celltype'] = ctx
    pr_curve = pd.concat([pr_curve,pr_table],axis=0)
    res_summary.append(['Pearson',round(pr_auc,5),round(epr,5),round(f1,5)])
    # pr_summary.append(["Pearson",t_gold, round(pr_auc,5)])
    # epr_summary.append(["Pearson",t_gold, round(epr,5)])
    for soft in soft_res.keys():
    # for soft in ['FigR']:

    # for soft in spa_res.keys():
    #     gene_peak_res = spa_res[soft]
        tf_gene_res = soft_res[soft]['grn_res']
        if tf_gene_res is not None:
            tf_gene_res = tf_gene_res[["TF","Gene","Score"]]
            tf_gene_res = tf_gene_res.groupby(['TF', 'Gene'])['Score'].max().reset_index() #只保留最高的。
            
            # tf_gene_res2 = tf_gene_res.nlargest(10000,"Score")
            tf_gene_res2 = tf_gene_res[tf_gene_res['TF'].isin(sel_tf)]
            tf_gene_res2 =  tf_gene_res2.nlargest(10000,"Score")
            print(f"{t_gold}: {soft}- candidates : {tf_gene_res2.shape[0]}")
            if tf_gene_res2.shape[0] > 1:
                pr_auc,epr,f1, pr_table = eval_tf_gene(grn_res=tf_gene_res2,gold_data=tf_knock,
                            all_comb=tf_baseline2.shape[0],label=soft ) # 
                # pr_table['celltype'] = ctx
            else:
                pr_auc = 0
                epr = 0
                f1 = 0
                pr_table = None
            pr_curve = pd.concat([pr_curve,pr_table],axis=0)
            res_summary.append([soft,round(pr_auc,5),round(epr,5),round(f1,5)])

res_summary = pd.DataFrame(res_summary)
res_summary.columns = ["method","PRAUC","EPR","F1"]

/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:536: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  soft_pred['pair'] = soft_pred.apply(lambda row: f"{row[0]}_{row[1]}", axis=1)


PBMC_TF_knock: SCENIC+- candidates : 425
PBMC_TF_knock: LINGER- candidates : 0
PBMC_TF_knock: linger_thres_scale2_samp- candidates : 421
PBMC_TF_knock: scenic2_thres_scale2_samp- candidates : 212


/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:536: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  soft_pred['pair'] = soft_pred.apply(lambda row: f"{row[0]}_{row[1]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:536: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  soft_pred['pair'] = soft_pred.apply(lambda row: f"{row[0]}_{row[1]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:536: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, 

In [49]:
res_summary

,method,PRAUC,EPR,F1
0,Pearson,0.03870,0.04459,0.06652
1,SCENIC+,0.05267,0.00407,0.06652
2,LINGER,0.00000,0.00000,0.00000
3,linger_thres_scale2_samp,0.07314,0.00637,0.06652
4,scenic2_thres_scale2_samp,0.04314,0.00149,0.06652


In [50]:
pr_curve

,prec,recall,prauc,method
0,0.034403,1.000000,0.038704,Pearson
1,0.041700,0.056519,0.038704,Pearson
2,0.041704,0.056519,0.038704,Pearson
3,0.041708,0.056519,0.038704,Pearson
4,0.041713,0.056519,0.038704,Pearson
...,...,...,...,...
209,0.000000,0.000000,0.043144,scenic2_thres_scale2_samp
210,0.000000,0.000000,0.043144,scenic2_thres_scale2_samp
211,0.000000,0.000000,0.043144,scenic2_thres_scale2_samp
212,0.000000,0.000000,0.043144,scenic2_thres_scale2_samp


In [51]:
res_summary.to_csv(f"{outdir}/pbmc_tf_knock_res.csv",index=False,header=True)
pr_curve.to_csv(f"{outdir}/pbmc_tf_knock_pr_curve.csv",index=False,header=True)